# 11장 사전학습과 글 만들기

- 학습하는 셀은 GPU에서 실행합니다(런타임 → 런타임 유형 변경 → T4 GPU). GPU가 없으면 학습 셀을 건너뛰세요. 다음 셀에서 미리 학습시켜 둔 파일을 내려받아 이어 갑니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

데이터: [Tiny-Ko-Stories](https://huggingface.co/datasets/psymon/Tiny-Ko-Stories) (psymon, CC BY 4.0) · 모델: [JosephLee/modu-llm](https://huggingface.co/JosephLee/modu-llm) (CC BY 4.0)

## 실습 11-1 준비

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import os
import torch
from torch.nn import functional as F
from modu_llm import Vocab, GPT, GPTConfig, get_batch, generate
from modu_llm import prepare_stories, load_checkpoint, get_file, train, mount_drive
from modu_llm import get_device, plot_history, show

device = get_device(need_gpu=True)

## 실습 11-2 숫자 하나짜리 모델

**먼저 예상해 보세요.** w를 0부터 4까지 바꾸면 손실은 어디에서 가장 작을까요? 3을 넘어가면 어떻게 될까요?

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])          # 문제
y = 3 * x                                  # 정답: 문제의 3배

def loss_at(w):
    return ((w * x - y) ** 2).mean()

for w in [0.0, 1.0, 2.0, 3.0, 4.0]:
    print(f"w = {w}: 손실 {loss_at(w):.2f}")

## 실습 11-3 발밑의 비탈, 기울기

**먼저 예상해 보세요.** w = 0에서 w를 조금 늘리면 손실은 커질까요, 작아질까요?

In [ ]:
slope = (loss_at(0.01) - loss_at(0.0)) / 0.01
print(f"어림한 기울기: {slope:.2f}")

w = torch.tensor(0.0, requires_grad=True)
loss = ((w * x - y) ** 2).mean()
loss.backward()
print(f"backward로 구한 기울기: {w.grad:.2f}")

## 실습 11-4 기울기의 반대쪽으로 한 번 고치기

**먼저 예상해 보세요.** 이렇게 학습을 다섯 번 하면 w는 3에 얼마나 가까워질까요?

In [ ]:
w = torch.tensor(0.0, requires_grad=True)  # 고칠 숫자 하나, 0에서 시작
lr = 0.05                                  # 학습률(한 번에 고치는 크기)

for i in range(5):
    loss = ((w * x - y) ** 2).mean()       # 예측하고 손실 재기
    loss.backward()                        # 기울기 구하기
    print(f"{i + 1}번째: w {w.item():.3f} | 손실 {loss.item():.3f} | 기울기 {w.grad.item():.3f}")
    with torch.no_grad():
        w -= lr * w.grad                   # 기울기 반대쪽으로 고치기
    w.grad.zero_()                         # 다음 번을 위해 기울기 지우기
print(f"다섯 번 고친 뒤: w {w.item():.3f}")

## 실습 11-5 고치기는 옵티마이저에게

**먼저 예상해 보세요.** 학습률을 0.25로 크게 잡으면 3에 더 빨리 다가갈까요?

In [ ]:
def fit(lr, steps=5):
    w = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([w], lr=lr)
    for _ in range(steps):
        loss = ((w * x - y) ** 2).mean()   # ① 예측하고 손실 재기
        optimizer.zero_grad()              # ② 지난 기울기 지우기
        loss.backward()                    # ③ 기울기 구하기
        optimizer.step()                   # ④ 고치기
    return w.item()

for lr in [0.01, 0.05, 0.25]:
    print(f"학습률 {lr}: 다섯 번 고친 뒤 w = {fit(lr):.3f}")

## 실습 11-6 준비: 이야기 20만 편

In [ ]:
vocab = Vocab.build()
folder = mount_drive()
train_data, val_data = prepare_stories(200_000, vocab, folder)

## 실습 11-7 작은 GPT로 학습 다섯 번

**먼저 예상해 보세요.** 작은 GPT를 다섯 번 학습시키면 손실은 어떻게 될까요?

In [ ]:
model = GPT(GPTConfig()).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

for i in range(5):
    x, y = get_batch(train_data, 384, 32, device)
    _, loss = model(x, y)      # ① 예측하고 손실 재기
    optimizer.zero_grad()      # ② 지난 기울기 지우기
    loss.backward()            # ③ 기울기 구하기
    optimizer.step()           # ④ 고치기
    print(f"{i + 1}번째: 손실 {loss.item():.3f}")

## 실습 11-8 네 줄을 30분 동안 되풀이하기

In [ ]:
ckpt_path = f"{folder}/pretrain.pt"
if os.path.exists(ckpt_path):
    model, optimizer, step, history = load_checkpoint(ckpt_path, device, optimizer_lr=5e-4)
    print(f"{step}번 학습한 모델을 불러와 이어서 학습합니다.")
else:
    model = GPT(GPTConfig()).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
    step = 0
    history = []
    print("처음부터 학습합니다.")

step, history = train(model, optimizer, train_data, val_data,
                      minutes=30, ckpt_path=ckpt_path, eval_every=250,
                      start_step=step, history=history, vocab=vocab)
print(f"학습 끝: {step}번 학습 | 검증 손실 {history[-1]['val']:.3f}")

## 실습 11-9 손실 그래프 읽기

In [ ]:
model, _, step, history = load_checkpoint(get_file("pretrain.pt"), "cpu")
print(f"{step}번 학습한 모델")
plot_history(history)

## 실습 11-10 학습 전·후 글

**먼저 예상해 보세요.** 두 모델은 "어느 날, " 뒤에 각각 어떤 글을 쓸까요?

In [ ]:
torch.manual_seed(0)
before = GPT(GPTConfig())
for name, m in [("학습 전", before), ("학습 뒤", model)]:
    torch.manual_seed(0)
    story = generate(m, vocab, "어느 날, ", max_new_tokens=100)
    print(f"[{name}] 어느 날, {story}")
    print()

## 실습 11-11 온도

**먼저 예상해 보세요.** 점수를 0.5로 나누면 1위 글자의 확률은 커질까요, 작아질까요?

In [ ]:
idx = torch.tensor([vocab.encode("어느 날, ")])
with torch.no_grad():
    logits, _ = model(idx)
last = logits[0, -1]                        # 마지막 자리의 점수
top = torch.topk(last, 5).indices           # 점수 상위 5글자의 번호
table = []
for t in [0.5, 1.0, 2.0]:
    probs = F.softmax(last / t, dim=-1)     # 온도로 나눈 뒤 확률로
    table.append(probs[top] * 100)          # 상위 5글자의 확률(%)
show(torch.stack(table),
     rows=["온도 0.5", "온도 1.0", "온도 2.0"],
     cols=[vocab.chars[i] for i in top.tolist()])

## 실습 11-12 top-k와 직접 쓰는 글 만들기

**먼저 예상해 보세요.** 온도를 1.5로 높이면 글이 어떻게 달라질까요? 여기에 top-k 10을 더하면요?

In [ ]:
def write(prompt, length=80, temperature=1.0, top_k=None):
    ids = vocab.encode(prompt)
    for i in range(length):
        x = torch.tensor([ids[-384:]])                  # 마지막 384글자만
        with torch.no_grad():
            logits, _ = model(x)
        last = logits[0, -1] / temperature             # ① 온도로 나누기
        if top_k is not None:
            kth = torch.topk(last, top_k).values[-1]
            last = last.masked_fill(last < kth, float("-inf"))   # ② k등 아래는 가리기
        probs = F.softmax(last, dim=-1)
        next_id = torch.multinomial(probs, 1).item()   # ③ 확률대로 하나 뽑기
        ids.append(next_id)                            # ④ 붙이기
    return vocab.decode(ids)

for t, k in [(0.5, None), (1.5, None), (1.5, 10)]:
    torch.manual_seed(0)
    print(f"[온도 {t}, top_k {k}]", write("어느 날, ", temperature=t, top_k=k))
    print()